# ch03_2. Fashion-MNIST 간단 예측

## 개요

- 저장된 Fashion-MNIST MLP Checkpoint(체크포인트, 모델 저장 파일)를 불러옵니다.
- Fashion-MNIST 테스트 이미지 한 장을 예측합니다.

## 공통 설정

데이터·모델·이미지 폴더 위치를 지정합니다.

In [1]:
from pathlib import Path

DATA_DIR = Path("data")
MODEL_DIR = Path("models")
PNG_DIR = Path("png")

DATA_DIR.mkdir(exist_ok=True)

MODEL_DIR.mkdir(exist_ok=True)
PNG_DIR.mkdir(exist_ok=True)

print("데이터 폴더:", DATA_DIR)
print("모델 폴더  :", MODEL_DIR)
print("이미지 폴더:", PNG_DIR)


데이터 폴더: data
모델 폴더  : models
이미지 폴더: png


## 공통 import

In [2]:
from pathlib import Path
import torch
from torch import nn
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

### 1) 실행 장치 확인

```text
Fashion-MNIST 학습
→ 검증
→ 테스트 평가
→ Checkpoint 저장
```

```text
Checkpoint 불러오기
→ 같은 모델 구조 복원
→ 테스트 이미지 선택
→ 예측과 확률 확인
```

 모델을 다시 학습하지 않습니다.

따라서 다음 항목은 사용하지 않습니다.

- 손실함수
- Optimizer(옵티마이저, 가중치 갱신 방법)
- 역전파
- Epoch 반복

```text
Checkpoint 파일 준비
→ 모델 구조 생성
→ 저장된 가중치 적용
→ Fashion-MNIST 테스트 데이터 준비
→ 이미지 한 장 선택
→ Batch 차원 추가
→ 로짓 계산
→ Softmax 확률 계산
→ 예측 결과 확인
```

GPU를 사용할 수 있으면 CUDA(쿠다, 엔비디아 GPU 연산 환경)를 사용하고, 그렇지 않으면 CPU를 사용합니다.

In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("사용 장치:", device)

사용 장치: cuda


### 2) 저장한 Checkpoint 불러오기

저장된 파일 이름:

```text
ch03_1_best_fashion_mnist_mlp.pt
```

새 런타임이라면 저장해 둔 Checkpoint 파일을 업로드합니다.

In [4]:
checkpoint_path = MODEL_DIR / "ch03_1_best_fashion_mnist_mlp.pt"

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"{checkpoint_path.name} 파일이 없습니다. "
        "저장된 Fashion-MNIST MLP Checkpoint를 준비해 주세요."
    )

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

print("Checkpoint 불러오기 완료")

print(
    "저장된 검증 정확도:",
    f"{checkpoint['validation_accuracy']:.4f}"
)

print(
    "저장된 테스트 정확도:",
    f"{checkpoint['test_accuracy']:.4f}"
)

FileNotFoundError: ch04_5_best_fashion_mnist_mlp.pt 파일이 없습니다. 저장된 Fashion-MNIST MLP Checkpoint를 준비해 주세요.

### 3) 모델 구성과 가중치 복원

Checkpoint에는 학습된 가중치가 들어 있습니다.

가중치를 사용하려면 학습 때와 같은 모델 구조를 먼저 만들어야 합니다.

```text
Flatten
→ Linear(784, 256)
→ ReLU
→ Linear(256, 128)
→ ReLU
→ Linear(128, 10)
```

별도의 모델 클래스는 만들지 않고 `nn.Sequential`을 사용합니다.

In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("모델 복원 완료")

### 4) Fashion-MNIST 데이터셋 준비

예측할 때는 다음 코드를 먼저 실행합니다.

```python
model.eval()
```

`eval()`은 Evaluation Mode(이밸류에이션 모드, 평가 모드)로 전환합니다.

현재 MLP에는 Dropout이나 Batch Normalization이 없지만, 예측 코드에서는 항상 평가 모드로 전환하는 습관을 들이는 것이 좋습니다.

훈련 데이터는 사용하지 않습니다.

테스트 데이터에서 이미지 한 장을 선택하여 예측합니다.

In [ ]:
transform = transforms.ToTensor()

test_dataset = datasets.FashionMNIST(
    root=str(DATA_DIR / "torchvision"),
    train=False,
    transform=transform,
    download=True
)

class_names_ko = checkpoint[
    "class_names_ko"
]

print("테스트 데이터:", len(test_dataset))

print("의류 클래스 수:", len(class_names_ko))

### 5) 예측할 이미지 선택

`TEST_IMAGE_INDEX`를 바꾸면 다른 테스트 이미지를 예측할 수 있습니다.

```python
TEST_IMAGE_INDEX = 0
```

사용 가능한 범위는 `0`부터 `9999`까지입니다.

In [ ]:
TEST_IMAGE_INDEX = 0

image, true_label = test_dataset[
    TEST_IMAGE_INDEX
]

print("선택한 이미지 번호:", TEST_IMAGE_INDEX)

print("이미지 shape:", image.shape)

print(
    "실제 의류:",
    class_names_ko[true_label]
)

### 6) Batch 차원 추가

테스트 이미지 한 장의 shape:

```text
[1, 28, 28]
```

모델은 여러 이미지를 한 번에 받는 형태를 사용하므로 Batch(배치, 데이터 묶음) 차원을 추가합니다.

```text
[1, 28, 28]
→ unsqueeze(0)
→ [1, 1, 28, 28]
```

맨 앞의 `1`은 이미지가 한 장이라는 뜻입니다.

In [ ]:
input_image = image.unsqueeze(
    0
).to(device)

print(
    "모델 입력 shape:",
    input_image.shape
)

### 7) 이미지 한 장 예측

예측 순서:

```text
이미지
→ MLP
→ 의류별 로짓 10개
→ Softmax
→ 의류별 확률
→ argmax
→ 최종 예측
```

예측에서는 기울기가 필요하지 않으므로 `torch.no_grad()`를 사용합니다.

In [ ]:
with torch.no_grad():
    logits = model(
        input_image
    )

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    predicted_label = logits.argmax(
        dim=1
    ).item()

    predicted_probability = probabilities[
        0,
        predicted_label
    ].item()

print("로짓 shape:", logits.shape)

print(
    "실제 의류:",
    class_names_ko[true_label]
)

print(
    "예측 의류:",
    class_names_ko[predicted_label]
)

print(
    f"예측 확률: "
    f"{predicted_probability * 100:.2f}%"
)

print(
    "예측 결과:",
    "정답" if predicted_label == true_label else "오답"
)

### 8) 상위 3개 예측 확인

최종 예측 하나만 보면 모델이 어떤 의류 사이에서 고민했는지 알기 어렵습니다.

확률이 높은 의류 세 개를 함께 확인합니다.

In [ ]:
top_probabilities, top_labels = torch.topk(
    probabilities[0],
    k=3
)

print("상위 3개 예측")

for rank, (
    label,
    probability
) in enumerate(
    zip(
        top_labels,
        top_probabilities
    ),
    start=1
):
    label_index = label.item()

    print(
        f"{rank}위 | "
        f"{class_names_ko[label_index]} | "
        f"{probability.item() * 100:.2f}%"
    )

### 9) 이미지와 예측 결과 표시

제목에 다음 내용을 표시합니다.

- 실제 의류
- 예측 의류
- 예측 확률

In [ ]:
plt.figure(figsize=(4, 4))

plt.imshow(
    image.squeeze(),
    cmap="gray"
)

plt.title(
    f"실제: {class_names_ko[true_label]}\n"
    f"예측: {class_names_ko[predicted_label]} "
    f"({predicted_probability * 100:.2f}%)"
)

plt.axis("off")

plt.show()

### 10) 상위 3개 확률 그래프

가장 높은 막대가 최종 예측 의류입니다.

In [ ]:
top_names = [
    class_names_ko[label.item()]
    for label in top_labels
]

top_percentages = [
    probability.item() * 100
    for probability in top_probabilities
]

plt.figure(figsize=(7, 4))

plt.bar(
    top_names,
    top_percentages
)

plt.xlabel("Class")

plt.ylabel("Probability (%)")

plt.title("Top 3 Predictions")

plt.ylim(0, 100)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.show()

### 11) 8장을 묶어 한 번에 예측

테스트 데이터의 앞부분 8개를 한 번에 예측합니다.

한 장을 예측할 때와 차이는 Batch 크기뿐입니다.

```text
이미지 한 장
→ [1, 1, 28, 28]

이미지 여덟 장
→ [8, 1, 28, 28]
```

In [ ]:
images = torch.stack(
    [
        test_dataset[index][0]
        for index in range(8)
    ]
)

labels = torch.tensor(
    [
        test_dataset[index][1]
        for index in range(8)
    ]
)

with torch.no_grad():
    batch_logits = model(
        images.to(device)
    )

    batch_probabilities = torch.softmax(
        batch_logits,
        dim=1
    )

    batch_confidences, batch_predictions = (
        batch_probabilities.max(dim=1)
    )

batch_predictions = batch_predictions.cpu()

batch_confidences = batch_confidences.cpu()

### 12) 배치 예측 결과 출력

In [ ]:
for index in range(8):
    actual_name = class_names_ko[
        labels[index].item()
    ]

    predicted_name = class_names_ko[
        batch_predictions[index].item()
    ]

    print(
        f"{index}번 | "
        f"실제 {actual_name:<6} | "
        f"예측 {predicted_name:<6} | "
        f"{batch_confidences[index].item() * 100:.2f}%"
    )

## 핵심 정리

- 모델을 다시 학습하지 않습니다.
- `unsqueeze(0)`은 이미지 앞에 Batch 차원을 추가합니다.
- 여러 이미지 예측도 한 장 예측과 같은 흐름으로 수행합니다.